# Neutral Desk – realistic lip-sync anchor (free Colab GPU)

1. **Runtime → Change runtime type → T4 GPU** (free).
2. On your PC run `python run.py pack` → it creates `output/<date>/lipsync_input.zip` (your anchor face + voice files).
3. Run the cells below. Upload the zip when asked. Download `anchor_clips.zip` at the end.
4. Unzip it into `output/<date>/anchor/`, set `anchor.mode: lipsync` in `config.yaml`, run `python run.py video`.

Face image tips: a front-facing, fictional person (make one with Stable Diffusion / Fooocus), shoulders visible, plain background, 512×512 or larger. **Never use a real person's face.**

Uses [SadTalker](https://github.com/OpenTalker/SadTalker) (free, open source). Its dependencies age fast; if a cell fails, read the error — it's usually one version pin.

In [ ]:
# 1) install SadTalker (~3 min)
!git clone -q https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip -q install -r requirements.txt
!pip -q install "numpy<2"
# compatibility patch for newer torchvision
!sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/' $(python -c "import basicsr,os;print(os.path.dirname(basicsr.__file__))")/data/degradations.py || true
!bash scripts/download_models.sh

In [ ]:
# 2) upload lipsync_input.zip
from google.colab import files
import zipfile, glob, os
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('/content/in')
face = glob.glob('/content/in/face.*')[0]
wavs = sorted(glob.glob('/content/in/audio/*.wav'))
print(face, len(wavs), 'segments')

In [ ]:
# 3) generate one talking-head clip per segment (~1 min per minute of audio on T4)
import subprocess, shutil
os.makedirs('/content/clips', exist_ok=True)
for w in wavs:
    seg = os.path.splitext(os.path.basename(w))[0]
    res = f'/content/res/{seg}'
    subprocess.run(['python', 'inference.py', '--driven_audio', w, '--source_image', face,
                    '--result_dir', res, '--still', '--preprocess', 'full', '--enhancer', 'gfpgan'], check=True)
    mp4 = sorted(glob.glob(f'{res}/*.mp4'))[-1]
    shutil.copy(mp4, f'/content/clips/{seg}.mp4')
    print('done', seg)

In [ ]:
# 4) download the clips
shutil.make_archive('/content/anchor_clips', 'zip', '/content/clips')
files.download('/content/anchor_clips.zip')